# Trước tách module (M) so với bản hiện tại (H): `bridge-m-h-kaggle-v1`

Kaggle **CPU / Internet On**, rồi **Save Version → Save & Run All (Commit)** (khoảng 1 giờ; chạy commit để không bị ngắt vì idle).

- **M** = `48d01c3`: backend đơn khối, một DB, trước khi tách module.
- **H** = `e0e8937`: ba service (Core, Clinic, Notification), ba DB, RabbitMQ, Redis, Nginx; CQRS projection tắt (mặc định).
- **Tải:** bài PCN (100 người dùng chờ phản hồi, nghỉ 0,5–1,5 s; GET pets 4, appointments 2, medical 2, hotel 1, invoices 1, notifications 1; POST appointment 2, rescue 1). Lịch đặt theo lưới 30 phút 08:00–17:00, mỗi lần một slot riêng (H từ P10 không nhận giờ lẻ); báo thất lạc mỗi lần một số điện thoại riêng.
- **Thứ tự:** smoke M, smoke H (2 người, 15 s), rồi 4 lượt đo **M, H, H, M**, mỗi lượt warmup 30 s + đo 300 s trên dữ liệu mới. Thứ tự ABBA triệt tiêu độ chậm dần của Kaggle.
- **Khác cấu hình duy nhất:** ngưỡng limiter theo IP và rescue của H được nâng (mọi traffic chung một IP loopback); limiter vẫn chạy. M không có limiter ngoài `/api/auth`.
- **Cách đọc (chốt trước khi chạy):** tỷ lệ p95 H/M cân bằng thứ tự ≤ 0,9 là H nhanh hơn, 0,9–1,1 tương đương, ≥ 1,1 H chậm hơn. Cùng ngưỡng cho CPU server mỗi request thành công.

Không retry, không đổi tải. Gửi lại file ZIP ở cuối để review.

In [ ]:
MODE = 'full'   # 'full' (đo) | 'smoke' (chỉ kiểm cài đặt, khoảng 20 phút)
assert MODE in ('full', 'smoke')
import base64, hashlib, importlib.util, io, json, os, platform, sys, uuid
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
assert platform.system() == 'Linux' and Path('/kaggle/working').is_dir(), 'Chạy trong notebook Kaggle.'
assert os.geteuid() == 0, 'Cần quyền root của kernel.'

SOURCE_SHA256 = '19dfd581f1a9eab2090fc900f335cdeb2fed3a961dbbcceb2736ebd6351e4cfb'
VALIDATOR_BASE64 = 'IiIiUGFjayBNIGFuZCBIIGFuZCB3cml0ZSB0aGUgYnJpZGdlLW0taC1rYWdnbGUtdjEgbm90ZWJvb2sgKG5vdCBleGVjdXRlZCkuCgpNIGlzIGNvbW1pdCA0OGQwMWMzICh0aGUgbW9ub2xpdGggdGhlIDI5LzA5IG5pcG9wZXRvIHJ1biBtZWFzdXJlZCkgcGx1cyBvbmUgaGFybmVzcwpmaWxlIGl0IHByZWRhdGVzIChwY25fY2hlY2tzLm1qcywgdGhlIHBoYXNlLTEgUENOIHNlZWQpLiBIIGlzIGEgY29tbWl0dGVkIHJldmlzaW9uCm9mIHRoaXMgcmVwb3NpdG9yeSAoZGVmYXVsdCBIRUFEKTsgdGhlIHdvcmtpbmcgdHJlZSBtdXN0IG5vdCBkaWZmZXIgZnJvbSBpdCBpbiBhbnkKcGFja2VkIHBhdGgsIHNvIEggaXMgZXhhY3RseSB0aGF0IGNvbW1pdC4gT25seSBidWlsZC9ydW50aW1lIHNvdXJjZXMgYW5kIHRoZQpiZW5jaG1hcmsgaGFybmVzcyBhcmUgcGFja2VkOiBubyAuZ2l0LCBlbnYgZmlsZXMsIGtleXMsIG5vdGVib29rcywgZGlzdCBvcgpub2RlX21vZHVsZXMuIFRoZSBhcmNoaXZlIGlzIGVtYmVkZGVkIGluIHRoZSBub3RlYm9vayAodW5kZXIgS2FnZ2xlJ3MgMSBNQgpTYXZlIFZlcnNpb24gbGltaXQpLCBzbyB0aGUgdXNlciBvbmx5IGltcG9ydHMgdGhlIG5vdGVib29rIGFuZCBydW5zIGl0LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGJhc2U2NAppbXBvcnQgaGFzaGxpYgppbXBvcnQgaW8KaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoLCBQdXJlUG9zaXhQYXRoCmltcG9ydCBzdGF0CmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCB0YXJmaWxlCmltcG9ydCB6aXBmaWxlCgpSRVBPID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudHNbM10KU1RVRFkgPSAnYnJpZGdlLW0taC1rYWdnbGUtdjEnCkZPUk1BVCA9ICdicmlkZ2UtbS1oLXNvdXJjZS0xJwpNX0NPTU1JVCA9ICc0OGQwMWMzYzk1ZDEzNmY4MDhiODQ2YTY2YTc1ZmUwOGFhY2Q5MmJlJwpNX1BBVEhTID0gKCdiYWNrZW5kJywgJ2JlbmNobWFya3Mva2FnZ2xlL3ByZXBhcmVfZml4dHVyZS5tanMnLCAnYmVuY2htYXJrcy9rYWdnbGUvZml4dHVyZURhdGEubWpzJywKICAgICAgICAgICAnZnJvbnRlbmQvcGFja2FnZS5qc29uJywgJ3BhY2thZ2UuanNvbicsICdwYWNrYWdlLWxvY2suanNvbicpCk1fT1ZFUkxBWSA9ICgnYmVuY2htYXJrcy9rYWdnbGUvcGNuX2NoZWNrcy5tanMnLCkKSEFSTkVTUyA9IHR1cGxlKCdiZW5jaG1hcmtzL3BoYXNlMi9rYWdnbGUvJyArIG5hbWUgZm9yIG5hbWUgaW4gKAogICAgJ2JyaWRnZV9zdHVkeS5weScsICdicmlkZ2VfbW9ub2xpdGgucHknLCAnYnJpZGdlX2xvY3VzdC5weScsICdtYWtlX2JyaWRnZV9ub3RlYm9vay5weScsICdwY25fcnVubmVyLnB5JywgJ3Bjbl9sb2N1c3QucHknLAogICAgJ3Bjbl9hZGFwdGVyLnB5JywgJ3Bjbl9kYXRhc2V0Lm1qcycsICduYXRpdmVfc3RhY2sucHknLCAnc3RhY2subWpzJywgJ2JlbmNobWFyay5weScsICdhY2NlcHRhbmNlLmpzb24nLCAnY3Fyc19wcm9iZS5weScsCiAgICAnY3Fyc19kaWFnbm9zdGljcy5weScpKSArIHR1cGxlKCdiZW5jaG1hcmtzL2thZ2dsZS8nICsgbmFtZSBmb3IgbmFtZSBpbiAoCiAgICAncGNuX2xvY3VzdGZpbGUucHknLCAncGNuX2NvbnRyYWN0LnB5JywgJ21ldHJpY3MucHknLCAncGNuX2NoZWNrcy5tanMnKSkgKyAoJ2JlbmNobWFya3MvcGhhc2UyL2RhdGFzZXQubWpzJywpCiMgcGhhc2UyOmJ1aWxkIHJ1bnMgdGhlc2UgYWZ0ZXIgdHNjIChnZW5lcmF0ZWQgUHJpc21hIGNsaWVudHMgaW50byBkaXN0KS4KQlVJTERfVE9PTFMgPSAoJ3Rvb2xzL3BoYXNlMi9jb3B5LWNvcmUtY2xpZW50Lm1qcycsICd0b29scy9waGFzZTIvY29weS1jbGllbnQubWpzJykKSF9QQVRIUyA9ICgnYmFja2VuZCcsICdzZXJ2aWNlcycsICdwYWNrYWdlcycsICpCVUlMRF9UT09MUywgKkhBUk5FU1MsICdmcm9udGVuZC9wYWNrYWdlLmpzb24nLCAncGFja2FnZS5qc29uJywgJ3BhY2thZ2UtbG9jay5qc29uJykKIyBCdWlsZHMgY29tcGlsZSBzcmMgb25seTsgdGVzdCBzdWl0ZXMgc3RheSBvdXQgb2YgdGhlIGFyY2hpdmUuClRFU1RfRElSUyA9ICgndGVzdHMnLCAndGVzdCcpClRBUkdFVCA9IFJFUE8gLyAnYmVuY2htYXJrcy9waGFzZTIvYnJpZGdlLW0taC1rYWdnbGUtdjEuaXB5bmInCkxJTUlUID0gMV8wMDBfMDAwCgoKZGVmIGRpZ2VzdCh2YWx1ZSk6CiAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYodmFsdWUpLmhleGRpZ2VzdCgpCgoKZGVmIGdpdCgqYXJncyk6CiAgICByZXR1cm4gc3VicHJvY2Vzcy5jaGVja19vdXRwdXQoWydnaXQnLCAqYXJnc10sIGN3ZD1SRVBPKQoKCmRlZiBzYWZlKG5hbWUpOgogICAgcCA9IFB1cmVQb3NpeFBhdGgobmFtZSkKICAgIGlmICgnXFwnIGluIG5hbWUgb3IgJzonIGluIG5hbWUgb3IgcC5pc19hYnNvbHV0ZSgpIG9yIHN0cihwKSAhPSBuYW1lIG9yICcuLicgaW4gcC5wYXJ0cwogICAgICAgICAgICBvciBhbnkocGFydCBpbiAoJy5naXQnLCAnbm9kZV9tb2R1bGVzJywgJ2Rpc3QnLCAnZ2VuZXJhdGVkJywgJ3VwbG9hZHMnKSBmb3IgcGFydCBpbiBwLnBhcnRzKQogICAgICAgICAgICBvciBwLm5hbWUuc3RhcnRzd2l0aCgnLmVudicpIG9yIHAuc3VmZml4IGluICgnLnBlbScsICcua2V5JywgJy5pcHluYicsICcuemlwJywgJy5iaW4nKSk6CiAgICAgICAgcmV0dXJuIEZhbHNlCiAgICByZXR1cm4gVHJ1ZQoKCmRlZiB0cmVlKGNvbW1pdCwgcGF0aHMpOgogICAgcmF3ID0gZ2l0KCdhcmNoaXZlJywgJy0tZm9ybWF0PXRhcicsIGNvbW1pdCwgKnBhdGhzKQogICAgZmlsZXMgPSB7fQogICAgd2l0aCB0YXJmaWxlLm9wZW4oZmlsZW9iaj1pby5CeXRlc0lPKHJhdykpIGFzIGFyY2hpdmU6CiAgICAgICAgZm9yIGluZm8gaW4gYXJjaGl2ZS5nZXRtZW1iZXJzKCk6CiAgICAgICAgICAgIGlmIGluZm8uaXNmaWxlKCkgYW5kIHNhZmUoaW5mby5uYW1lKSBhbmQgbm90IHNldChQdXJlUG9zaXhQYXRoKGluZm8ubmFtZSkucGFydHNbOi0xXSkgJiBzZXQoVEVTVF9ESVJTKToKICAgICAgICAgICAgICAgIGZpbGVzW2luZm8ubmFtZV0gPSBhcmNoaXZlLmV4dHJhY3RmaWxlKGluZm8pLnJlYWQoKQogICAgICAgICAgICBlbGlmIGluZm8uaXNzeW0oKSBvciBpbmZvLmlzbG5rKCk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdCUklER0VfTElOS19JTl9UUkVFOiAnICsgaW5mby5uYW1lKQogICAgcmV0dXJuIGZpbGVzCgoKZGVmIHBhY2soaF9jb21taXQ9J0hFQUQnLCB3b3JrdHJlZV9oYXJuZXNzPUZhbHNlKToKICAgICIiIndvcmt0cmVlX2hhcm5lc3M6IGxvY2FsIGRyeSBydW5zIG9ubHk7IHRoZSBoYXJuZXNzIGNvbWVzIGZyb20gdGhlIHdvcmtpbmcgdHJlZSBhbmQgdGhlIG1hbmlmZXN0IHNheXMgc28uIiIiCiAgICBoID0gZ2l0KCdyZXYtcGFyc2UnLCBoX2NvbW1pdCkuZGVjb2RlKCkuc3RyaXAoKQogICAgIyBNYXJrZG93biBzdGF5cyBsb2NhbCBieSByZXBvc2l0b3J5IHBvbGljeSBhbmQgaXMgbmV2ZXIgcGFja2VkLgogICAgZGlydHkgPSAnXG4nLmpvaW4obGluZSBmb3IgbGluZSBpbiBnaXQoJ3N0YXR1cycsICctLXBvcmNlbGFpbicsICctLXVudHJhY2tlZC1maWxlcz1hbGwnLCAnLS0nLCAqSF9QQVRIUykuZGVjb2RlKCkuc3BsaXRsaW5lcygpCiAgICAgICAgICAgICAgICAgICAgICBpZiBub3QgbGluZS5lbmRzd2l0aCgnLm1kJykgYW5kIG5vdCAod29ya3RyZWVfaGFybmVzcyBhbmQgbGluZVszOl0gaW4gSEFSTkVTUykpCiAgICBpZiBoX2NvbW1pdCA9PSAnSEVBRCcgYW5kIGRpcnR5OgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0JSSURHRV9IX1dPUktUUkVFX0RJRkZFUlNfRlJPTV9IRUFEOlxuJyArIGRpcnR5KQogICAgbV9maWxlcywgaF9maWxlcyA9IHRyZWUoTV9DT01NSVQsIE1fUEFUSFMpLCB0cmVlKGgsIEhfUEFUSFMpCiAgICBpZiB3b3JrdHJlZV9oYXJuZXNzOgogICAgICAgIGhfZmlsZXMudXBkYXRlKHtuYW1lOiAoUkVQTyAvIG5hbWUpLnJlYWRfYnl0ZXMoKSBmb3IgbmFtZSBpbiBIQVJORVNTfSkKICAgIGZvciBuYW1lIGluIE1fT1ZFUkxBWToKICAgICAgICBpZiBuYW1lIGluIG1fZmlsZXM6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ0JSSURHRV9NX09WRVJMQVlfRVhJU1RTOiAnICsgbmFtZSkKICAgICAgICBtX2ZpbGVzW25hbWVdID0gaF9maWxlc1tuYW1lXQogICAgcGF5bG9hZCA9IHsqKnsnTS8nICsgazogdiBmb3IgaywgdiBpbiBtX2ZpbGVzLml0ZW1zKCl9LCAqKnsnSC8nICsgazogdiBmb3IgaywgdiBpbiBoX2ZpbGVzLml0ZW1zKCl9fQogICAgZmlsZXMgPSBbeydwYXRoJzogbmFtZSwgJ2J5dGVzJzogbGVuKHZhbHVlKSwgJ3NoYTI1Nic6IGRpZ2VzdCh2YWx1ZSl9IGZvciBuYW1lLCB2YWx1ZSBpbiBzb3J0ZWQocGF5bG9hZC5pdGVtcygpKV0KICAgIG1hbmlmZXN0ID0geydmb3JtYXQnOiBGT1JNQVQsICdzdHVkeSc6IFNUVURZLCAnbUNvbW1pdCc6IE1fQ09NTUlULCAnaENvbW1pdCc6IGgsICdtT3ZlcmxheSc6IGxpc3QoTV9PVkVSTEFZKSwKICAgICAgICAgICAgICAgICdoYXJuZXNzRnJvbVdvcmt0cmVlJzogd29ya3RyZWVfaGFybmVzcywKICAgICAgICAgICAgICAgICdmaWxlcyc6IGZpbGVzLCAndHJlZVNoYTI1Nic6IGRpZ2VzdChqc29uLmR1bXBzKGZpbGVzLCBzZXBhcmF0b3JzPSgnLCcsICc6JykpLmVuY29kZSgpKSwKICAgICAgICAgICAgICAgICdzY29wZSc6ICdCdWlsZC9ydW50aW1lIHNvdXJjZXMgYW5kIGJlbmNobWFyayBoYXJuZXNzIG9ubHk7IEggaXMgdGhlIGNvbW1pdHRlZCByZXZpc2lvbiwgTSBpcyA0OGQwMWMzIHBsdXMgdGhlIFBDTiBzZWVkLid9CiAgICBidWZmZXIgPSBpby5CeXRlc0lPKCkKICAgIHdpdGggemlwZmlsZS5aaXBGaWxlKGJ1ZmZlciwgJ3cnLCB6aXBmaWxlLlpJUF9ERUZMQVRFRCwgY29tcHJlc3NsZXZlbD05KSBhcyBhcmNoaXZlOgogICAgICAgIGZvciBuYW1lLCB2YWx1ZSBpbiBbKCdidW5kbGUtbWFuaWZlc3QuanNvbicsIChqc29uLmR1bXBzKG1hbmlmZXN0LCBpbmRlbnQ9MikgKyAnXG4nKS5lbmNvZGUoKSksICpzb3J0ZWQocGF5bG9hZC5pdGVtcygpKV06CiAgICAgICAgICAgIGluZm8gPSB6aXBmaWxlLlppcEluZm8obmFtZSwgKDIwMjYsIDEwLCAyOCwgMCwgMCwgMCkpCiAgICAgICAgICAgIGluZm8uZXh0ZXJuYWxfYXR0ciA9IChzdGF0LlNfSUZSRUcgfCAwbzYwMCkgPDwgMTYKICAgICAgICAgICAgaW5mby5jb21wcmVzc190eXBlID0gemlwZmlsZS5aSVBfREVGTEFURUQKICAgICAgICAgICAgYXJjaGl2ZS53cml0ZXN0cihpbmZvLCB2YWx1ZSkKICAgIHJldHVybiBidWZmZXIuZ2V0dmFsdWUoKSwgbWFuaWZlc3QKCgpkZWYgdmVyaWZpZWRfbWVtYmVycyhyYXcsIGV4cGVjdGVkKToKICAgICIiIlB1cmUgdmFsaWRhdG9yLCBhbHNvIGV4ZWN1dGVkIGJ5IHRoZSBub3RlYm9vayBiZWZvcmUgYW55dGhpbmcgaXMgd3JpdHRlbi4iIiIKICAgIGlmIGhhc2hsaWIuc2hhMjU2KHJhdykuaGV4ZGlnZXN0KCkgIT0gZXhwZWN0ZWQ6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQlJJREdFX0FSQ0hJVkVfSEFTSCcpCiAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShpby5CeXRlc0lPKHJhdykpIGFzIGFyY2hpdmU6CiAgICAgICAgaW5mb3MgPSBhcmNoaXZlLmluZm9saXN0KCkKICAgICAgICBpZiBsZW4oaW5mb3MpID4gMzAwMCBvciBsZW4oe2kuZmlsZW5hbWUgZm9yIGkgaW4gaW5mb3N9KSAhPSBsZW4oaW5mb3MpIG9yIHN1bShpLmZpbGVfc2l6ZSBmb3IgaSBpbiBpbmZvcykgPiA0MF8wMDBfMDAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdCUklER0VfQVJDSElWRV9TSEFQRScpCiAgICAgICAgZm9yIGluZm8gaW4gaW5mb3M6CiAgICAgICAgICAgIGlmIGluZm8uaXNfZGlyKCkgb3Igc3RhdC5TX0lTTE5LKGluZm8uZXh0ZXJuYWxfYXR0ciA+PiAxNik6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdCUklER0VfTElOS19PUl9ESVJFQ1RPUlknKQogICAgICAgIG1hbmlmZXN0ID0ganNvbi5sb2FkcyhhcmNoaXZlLnJlYWQoJ2J1bmRsZS1tYW5pZmVzdC5qc29uJykpCiAgICAgICAgZmlsZXMgPSBtYW5pZmVzdFsnZmlsZXMnXQogICAgICAgIGlmIG1hbmlmZXN0Wydmb3JtYXQnXSAhPSBGT1JNQVQgb3IgaGFzaGxpYi5zaGEyNTYoanNvbi5kdW1wcyhmaWxlcywgc2VwYXJhdG9ycz0oJywnLCAnOicpKS5lbmNvZGUoKSkuaGV4ZGlnZXN0KCkgIT0gbWFuaWZlc3RbJ3RyZWVTaGEyNTYnXToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignQlJJREdFX1RSRUVfSEFTSCcpCiAgICAgICAgaWYge2kuZmlsZW5hbWUgZm9yIGkgaW4gaW5mb3N9ICE9IHsnYnVuZGxlLW1hbmlmZXN0Lmpzb24nLCAqKHJbJ3BhdGgnXSBmb3IgciBpbiBmaWxlcyl9OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdCUklER0VfSU5WRU5UT1JZJykKICAgICAgICB2YWx1ZXMgPSB7fQogICAgICAgIGZvciByb3cgaW4gZmlsZXM6CiAgICAgICAgICAgIGdyb3VwLCBfLCBuYW1lID0gcm93WydwYXRoJ10ucGFydGl0aW9uKCcvJykKICAgICAgICAgICAgaWYgZ3JvdXAgbm90IGluICgnTScsICdIJykgb3Igbm90IHNhZmUobmFtZSk6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdCUklER0VfVU5TQUZFX1BBVEgnKQogICAgICAgICAgICB2YWx1ZSA9IGFyY2hpdmUucmVhZChyb3dbJ3BhdGgnXSkKICAgICAgICAgICAgaWYgbGVuKHZhbHVlKSAhPSByb3dbJ2J5dGVzJ10gb3IgaGFzaGxpYi5zaGEyNTYodmFsdWUpLmhleGRpZ2VzdCgpICE9IHJvd1snc2hhMjU2J106CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdCUklER0VfRklMRV9IQVNIJykKICAgICAgICAgICAgdmFsdWVzW3Jvd1sncGF0aCddXSA9IHZhbHVlCiAgICAgICAgcmV0dXJuIG1hbmlmZXN0LCB2YWx1ZXMKCgpkZWYgYnVpbGQodGFyZ2V0PVRBUkdFVCwgaF9jb21taXQ9J0hFQUQnLCB3b3JrdHJlZV9oYXJuZXNzPUZhbHNlKToKICAgIHJhdywgbWFuaWZlc3QgPSBwYWNrKGhfY29tbWl0LCB3b3JrdHJlZV9oYXJuZXNzKQogICAgZXhwZWN0ZWQgPSBkaWdlc3QocmF3KQogICAgdmVyaWZpZWRfbWVtYmVycyhyYXcsIGV4cGVjdGVkKQogICAgdmFsaWRhdG9yID0gUGF0aChfX2ZpbGVfXykucmVhZF9ieXRlcygpCiAgICBjZWxscyA9IFtdCgogICAgZGVmIGNlbGwoa2luZCwgdGV4dCk6CiAgICAgICAgaXRlbSA9IHsnY2VsbF90eXBlJzoga2luZCwgJ2lkJzogZidicmlkZ2UtdjEte2xlbihjZWxscyl9JywgJ21ldGFkYXRhJzoge30sICdzb3VyY2UnOiB0ZXh0LnN0cmlwKCkuc3BsaXRsaW5lcyhrZWVwZW5kcz1UcnVlKX0KICAgICAgICBpZiBraW5kID09ICdjb2RlJzoKICAgICAgICAgICAgaXRlbS51cGRhdGUoZXhlY3V0aW9uX2NvdW50PU5vbmUsIG91dHB1dHM9W10pCiAgICAgICAgY2VsbHMuYXBwZW5kKGl0ZW0pCiAgICBjZWxsKCdtYXJrZG93bicsIGYnJycjIFRyxrDhu5tjIHTDoWNoIG1vZHVsZSAoTSkgc28gduG7m2kgYuG6o24gaGnhu4duIHThuqFpIChIKTogYHtTVFVEWX1gCgpLYWdnbGUgKipDUFUgLyBJbnRlcm5ldCBPbioqLCBy4buTaSAqKlNhdmUgVmVyc2lvbiDihpIgU2F2ZSAmIFJ1biBBbGwgKENvbW1pdCkqKiAoa2hv4bqjbmcgMSBnaeG7nTsgY2jhuqF5IGNvbW1pdCDEkeG7gyBraMO0bmcgYuG7iyBuZ+G6r3QgdsOsIGlkbGUpLgoKLSAqKk0qKiA9IGB7TV9DT01NSVRbOjddfWA6IGJhY2tlbmQgxJHGoW4ga2jhu5FpLCBt4buZdCBEQiwgdHLGsOG7m2Mga2hpIHTDoWNoIG1vZHVsZS4KLSAqKkgqKiA9IGB7bWFuaWZlc3RbJ2hDb21taXQnXVs6N119YDogYmEgc2VydmljZSAoQ29yZSwgQ2xpbmljLCBOb3RpZmljYXRpb24pLCBiYSBEQiwgUmFiYml0TVEsIFJlZGlzLCBOZ2lueDsgQ1FSUyBwcm9qZWN0aW9uIHThuq90ICht4bq3YyDEkeG7i25oKS4KLSAqKlThuqNpOioqIGLDoGkgUENOICgxMDAgbmfGsOG7nWkgZMO5bmcgY2jhu50gcGjhuqNuIGjhu5NpLCBuZ2jhu4kgMCw14oCTMSw1IHM7IEdFVCBwZXRzIDQsIGFwcG9pbnRtZW50cyAyLCBtZWRpY2FsIDIsIGhvdGVsIDEsIGludm9pY2VzIDEsIG5vdGlmaWNhdGlvbnMgMTsgUE9TVCBhcHBvaW50bWVudCAyLCByZXNjdWUgMSkuIEzhu4tjaCDEkeG6t3QgdGhlbyBsxrDhu5tpIDMwIHBow7p0IDA4OjAw4oCTMTc6MDAsIG3hu5dpIGzhuqduIG3hu5l0IHNsb3QgcmnDqm5nIChIIHThu6sgUDEwIGtow7RuZyBuaOG6rW4gZ2nhu50gbOG6uyk7IGLDoW8gdGjhuqV0IGzhuqFjIG3hu5dpIGzhuqduIG3hu5l0IHPhu5EgxJFp4buHbiB0aG/huqFpIHJpw6puZy4KLSAqKlRo4bupIHThu7E6Kiogc21va2UgTSwgc21va2UgSCAoMiBuZ8aw4budaSwgMTUgcyksIHLhu5NpIDQgbMaw4bujdCDEkW8gKipNLCBILCBILCBNKiosIG3hu5dpIGzGsOG7o3Qgd2FybXVwIDMwIHMgKyDEkW8gMzAwIHMgdHLDqm4gZOG7ryBsaeG7h3UgbeG7m2kuIFRo4bupIHThu7EgQUJCQSB0cmnhu4d0IHRpw6p1IMSR4buZIGNo4bqtbSBk4bqnbiBj4bunYSBLYWdnbGUuCi0gKipLaMOhYyBj4bqldSBow6xuaCBkdXkgbmjhuqV0OioqIG5nxrDhu6FuZyBsaW1pdGVyIHRoZW8gSVAgdsOgIHJlc2N1ZSBj4bunYSBIIMSRxrDhu6NjIG7Dom5nICht4buNaSB0cmFmZmljIGNodW5nIG3hu5l0IElQIGxvb3BiYWNrKTsgbGltaXRlciB24bqrbiBjaOG6oXkuIE0ga2jDtG5nIGPDsyBsaW1pdGVyIG5nb8OgaSBgL2FwaS9hdXRoYC4KLSAqKkPDoWNoIMSR4buNYyAoY2jhu5F0IHRyxrDhu5tjIGtoaSBjaOG6oXkpOioqIHThu7cgbOG7hyBwOTUgSC9NIGPDom4gYuG6sW5nIHRo4bupIHThu7Eg4omkIDAsOSBsw6AgSCBuaGFuaCBoxqFuLCAwLDnigJMxLDEgdMawxqFuZyDEkcawxqFuZywg4omlIDEsMSBIIGNo4bqtbSBoxqFuLiBDw7luZyBuZ8aw4buhbmcgY2hvIENQVSBzZXJ2ZXIgbeG7l2kgcmVxdWVzdCB0aMOgbmggY8O0bmcuCgpLaMO0bmcgcmV0cnksIGtow7RuZyDEkeG7lWkgdOG6o2kuIEfhu61pIGzhuqFpIGZpbGUgWklQIOG7nyBjdeG7kWkgxJHhu4MgcmV2aWV3LicnJykKICAgIGNlbGwoJ2NvZGUnLCAnJydNT0RFID0gJ2Z1bGwnICAgIyAnZnVsbCcgKMSRbykgfCAnc21va2UnIChjaOG7iSBraeG7g20gY8OgaSDEkeG6t3QsIGtob+G6o25nIDIwIHBow7p0KQphc3NlcnQgTU9ERSBpbiAoJ2Z1bGwnLCAnc21va2UnKQppbXBvcnQgYmFzZTY0LCBoYXNobGliLCBpbXBvcnRsaWIudXRpbCwgaW8sIGpzb24sIG9zLCBwbGF0Zm9ybSwgc3lzLCB1dWlkCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aCwgUHVyZVBvc2l4UGF0aApmcm9tIHppcGZpbGUgaW1wb3J0IFppcEZpbGUKYXNzZXJ0IHBsYXRmb3JtLnN5c3RlbSgpID09ICdMaW51eCcgYW5kIFBhdGgoJy9rYWdnbGUvd29ya2luZycpLmlzX2RpcigpLCAnQ2jhuqF5IHRyb25nIG5vdGVib29rIEthZ2dsZS4nCmFzc2VydCBvcy5nZXRldWlkKCkgPT0gMCwgJ0PhuqduIHF1eeG7gW4gcm9vdCBj4bunYSBrZXJuZWwuJwonJycgKyAnXG5TT1VSQ0VfU0hBMjU2ID0gJyArIHJlcHIoZXhwZWN0ZWQpICsgJ1xuVkFMSURBVE9SX0JBU0U2NCA9ICcgKyByZXByKGJhc2U2NC5iNjRlbmNvZGUodmFsaWRhdG9yKS5kZWNvZGUoKSkgKwogICAgICAgICAnXG5TT1VSQ0VfQkFTRTY0ID0gJyArIHJlcHIoYmFzZTY0LmI2NGVuY29kZShyYXcpLmRlY29kZSgpKSArICcnJwpyYXcgPSBiYXNlNjQuYjY0ZGVjb2RlKFNPVVJDRV9CQVNFNjQsIHZhbGlkYXRlPVRydWUpCmFzc2VydCBoYXNobGliLnNoYTI1NihyYXcpLmhleGRpZ2VzdCgpID09IFNPVVJDRV9TSEEyNTYsICdTb3VyY2UgYXJjaGl2ZSBoYXNoIG1pc21hdGNoJwpuYW1lc3BhY2UgPSB7J19fbmFtZV9fJzogJ2JyaWRnZV92YWxpZGF0b3InLCAnX19maWxlX18nOiAnL3RtcC9icmlkZ2UtdmFsaWRhdG9yL2JlbmNobWFya3MvcGhhc2UyL2thZ2dsZS9tYWtlX2JyaWRnZV9ub3RlYm9vay5weSd9CmV4ZWMoY29tcGlsZShiYXNlNjQuYjY0ZGVjb2RlKFZBTElEQVRPUl9CQVNFNjQpLCAnPGJyaWRnZS12YWxpZGF0b3I+JywgJ2V4ZWMnKSwgbmFtZXNwYWNlKQptYW5pZmVzdCwgdmFsdWVzID0gbmFtZXNwYWNlWyd2ZXJpZmllZF9tZW1iZXJzJ10ocmF3LCBTT1VSQ0VfU0hBMjU2KQpSVU4gPSBzdHIodXVpZC51dWlkNCgpKQpQUklWQVRFID0gUGF0aCgnL3RtcCcpIC8gKCdicmlkZ2UtJyArIFJVTik7IFBSSVZBVEUubWtkaXIobW9kZT0wbzcwMCkKT1VUUFVUID0gUGF0aCgnL2thZ2dsZS93b3JraW5nJykgLyAoJ2JyaWRnZS0nICsgUlVOKQpmb3IgbmFtZSwgdmFsdWUgaW4gdmFsdWVzLml0ZW1zKCk6CiAgICBkZXN0aW5hdGlvbiA9IFBSSVZBVEUuam9pbnBhdGgoKlB1cmVQb3NpeFBhdGgobmFtZSkucGFydHMpCiAgICBhc3NlcnQgZGVzdGluYXRpb24ucmVzb2x2ZSgpLmlzX3JlbGF0aXZlX3RvKFBSSVZBVEUucmVzb2x2ZSgpKQogICAgZGVzdGluYXRpb24ucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggZGVzdGluYXRpb24ub3BlbigneGInKSBhcyBoYW5kbGU6CiAgICAgICAgaGFuZGxlLndyaXRlKHZhbHVlKQooUFJJVkFURSAvICdidW5kbGUtbWFuaWZlc3QuanNvbicpLndyaXRlX2J5dGVzKGpzb24uZHVtcHMobWFuaWZlc3QsIGluZGVudD0yKS5lbmNvZGUoKSkKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQUklWQVRFIC8gJ0gvYmVuY2htYXJrcy9waGFzZTIva2FnZ2xlJykpCnNwZWMgPSBpbXBvcnRsaWIudXRpbC5zcGVjX2Zyb21fZmlsZV9sb2NhdGlvbignYnJpZGdlX3N0dWR5JywgUFJJVkFURSAvICdIL2JlbmNobWFya3MvcGhhc2UyL2thZ2dsZS9icmlkZ2Vfc3R1ZHkucHknKQpzdHVkeSA9IGltcG9ydGxpYi51dGlsLm1vZHVsZV9mcm9tX3NwZWMoc3BlYyk7IHNwZWMubG9hZGVyLmV4ZWNfbW9kdWxlKHN0dWR5KQpwcmludCh7J3J1bic6IFJVTiwgJ3N0dWR5Jzogc3R1ZHkuU1RVRFksICdNJzogbWFuaWZlc3RbJ21Db21taXQnXVs6N10sICdIJzogbWFuaWZlc3RbJ2hDb21taXQnXVs6N10sICdzb3VyY2VfdmVyaWZpZWQnOiBUcnVlLCAnbW9kZSc6IE1PREV9KQonJycpCiAgICBjZWxsKCdtYXJrZG93bicsICcjIyBDaOG6oXlcbkluIHRp4bq/biDEkeG7mSBt4buXaSAzMCBzLiBMb2csIERCIHbDoCB0b2tlbiBjaOG7iSBu4bqxbSB0cm9uZyBgL3RtcGA7IFpJUCBjaOG7iSBjaOG7qWEgc+G7kSDEkW8uJykKICAgIGNlbGwoJ2NvZGUnLCAnJydmcm9tIElQeXRob24uZGlzcGxheSBpbXBvcnQgZGlzcGxheSwgRmlsZUxpbmsKV09SSyA9IFBSSVZBVEUgLyAnd29yayc7IFdPUksubWtkaXIoKQpzdW1tYXJ5ID0gc3R1ZHkucnVuX3N0dWR5KFBSSVZBVEUgLyAnTScsIFBSSVZBVEUgLyAnSCcsIFdPUkssIE9VVFBVVCwgUlVOLCBtb2RlPU1PREUsIHByb2dyZXNzPWxhbWJkYSB0ZXh0OiBwcmludCh0ZXh0LCBmbHVzaD1UcnVlKSkKY29tcGFyaXNvbiA9IHN1bW1hcnkuZ2V0KCdjb21wYXJpc29uJywge30pCnByaW50KGpzb24uZHVtcHMoe2s6IHN1bW1hcnkuZ2V0KGspIGZvciBrIGluICgnc3RhdHVzJywgJ3Ntb2tlJywgJ2Vycm9yX3R5cGUnLCAnZXJyb3InKX0sIGluZGVudD0yLCBlbnN1cmVfYXNjaWk9RmFsc2UpKQppZiBjb21wYXJpc29uLmdldCgnZGVjaXNpb24nKSA9PSAnQ09NUEFSRUQnOgogICAgcHJpbnQoZiJ7J29wZXJhdGlvbic6MjJ9IHsncDk1IE0gKG1zKSc6PjE2fSB7J3A5NSBIIChtcyknOj4xNn0geydIL00gcDk1Jzo+OH0geydIL00gcDUwJzo+OH0gIMSR4buNYyIpCiAgICBmb3IgbmFtZSwgb3AgaW4gY29tcGFyaXNvblsnb3BlcmF0aW9ucyddLml0ZW1zKCk6CiAgICAgICAgbSwgaCA9ICgnIC8gJy5qb2luKGYne3Y6LjFmfScgZm9yIHYgaW4gb3BbJ3A5NV9tcyddW2tdKSBmb3IgayBpbiAoJ00nLCAnSCcpKQogICAgICAgIHByaW50KGYie25hbWU6MjJ9IHttOj4xNn0ge2g6PjE2fSB7b3BbJ3A5NV9yYXRpbyddOjguMmZ9IHtvcFsncDUwX3JhdGlvJ106OC4yZn0gIHtvcFsncmVhZGluZyddfSIpCiAgICBwcmludChqc29uLmR1bXBzKHsnb3ZlcmFsbF9yYXRpb3NfSF9vdmVyX00nOiBjb21wYXJpc29uWydvdmVyYWxsX3JhdGlvc19IX292ZXJfTSddLAogICAgICAgICAgICAgICAgICAgICAgJ292ZXJhbGxfcDk1JzogY29tcGFyaXNvblsnb3ZlcmFsbF9wOTVfcmVhZGluZyddLCAnY3B1X3Blcl9zdWNjZXNzJzogY29tcGFyaXNvblsnY3B1X3Blcl9zdWNjZXNzX3JlYWRpbmcnXX0sCiAgICAgICAgICAgICAgICAgICAgIGluZGVudD0yLCBlbnN1cmVfYXNjaWk9RmFsc2UpKQplbHNlOgogICAgcHJpbnQoanNvbi5kdW1wcyhjb21wYXJpc29uLCBpbmRlbnQ9MiwgZW5zdXJlX2FzY2lpPUZhbHNlKSkKZGlzcGxheShGaWxlTGluayhzdW1tYXJ5Wydkb3dubG9hZCddKSkKJycnKQogICAgY2VsbCgnbWFya2Rvd24nLCAnIyMgR+G7rWkga+G6v3QgcXXhuqNcblThuqNpIGZpbGUgYGJyaWRnZS08cnVuPi56aXBgIOG7nyB0csOqbiAoaG/hurdjIHRhYiBPdXRwdXQgY+G7p2EgYuG6o24gY29tbWl0KSB2w6AgZ+G7rWkgbOG6oWkgxJHhu4MgcmV2aWV3LicpCiAgICBub3RlYm9vayA9IHsnY2VsbHMnOiBjZWxscywgJ21ldGFkYXRhJzogeydrZXJuZWxzcGVjJzogeydkaXNwbGF5X25hbWUnOiAnUHl0aG9uIDMnLCAnbGFuZ3VhZ2UnOiAncHl0aG9uJywgJ25hbWUnOiAncHl0aG9uMyd9LAogICAgICAgICAgICAgICAgJ2xhbmd1YWdlX2luZm8nOiB7J25hbWUnOiAncHl0aG9uJ30sICdrYWdnbGUnOiB7J2FjY2VsZXJhdG9yJzogJ25vbmUnLCAnaXNJbnRlcm5ldEVuYWJsZWQnOiBUcnVlLCAnbGFuZ3VhZ2UnOiAncHl0aG9uJywKICAgICAgICAgICAgICAgICdzb3VyY2VUeXBlJzogJ25vdGVib29rJ30sICdicmlkZ2Vfc291cmNlX3NoYTI1Nic6IGV4cGVjdGVkfSwgJ25iZm9ybWF0JzogNCwgJ25iZm9ybWF0X21pbm9yJzogNX0KICAgIHZhbHVlID0gKGpzb24uZHVtcHMobm90ZWJvb2ssIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTEpICsgJ1xuJykuZW5jb2RlKCkKICAgIGlmIGxlbih2YWx1ZSkgPiBMSU1JVDoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYnQlJJREdFX05PVEVCT09LX1RPT19MQVJHRToge2xlbih2YWx1ZSl9IGJ5dGVzJykKICAgIHRhcmdldCA9IFBhdGgodGFyZ2V0KQogICAgdGFyZ2V0LndyaXRlX2J5dGVzKHZhbHVlKQogICAgcmVjb3JkID0geydzdHVkeSc6IFNUVURZLCAnbm90ZWJvb2snOiB0YXJnZXQubmFtZSwgJ25vdGVib29rQnl0ZXMnOiBsZW4odmFsdWUpLCAnbm90ZWJvb2tTaGEyNTYnOiBkaWdlc3QodmFsdWUpLAogICAgICAgICAgICAgICdzb3VyY2VBcmNoaXZlU2hhMjU2JzogZXhwZWN0ZWQsICdhcmNoaXZlQnl0ZXMnOiBsZW4ocmF3KSwgJ21Db21taXQnOiBNX0NPTU1JVCwgJ2hDb21taXQnOiBtYW5pZmVzdFsnaENvbW1pdCddLAogICAgICAgICAgICAgICdmaWxlcyc6IGxlbihtYW5pZmVzdFsnZmlsZXMnXSksICdleGVjdXRpb24nOiAnTk9UX1JVTid9CiAgICB0YXJnZXQud2l0aF9zdWZmaXgoJy5tYW5pZmVzdC5qc29uJykud3JpdGVfdGV4dChqc29uLmR1bXBzKHJlY29yZCwgaW5kZW50PTIpICsgJ1xuJywgZW5jb2Rpbmc9J3V0Zi04JykKICAgIHJldHVybiByZWNvcmQKCgppZiBfX25hbWVfXyA9PSAnX19tYWluX18nOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0taC1jb21taXQnLCBkZWZhdWx0PSdIRUFEJykKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tdGFyZ2V0JywgdHlwZT1QYXRoLCBkZWZhdWx0PVRBUkdFVCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0td29ya3RyZWUtaGFybmVzcycsIGFjdGlvbj0nc3RvcmVfdHJ1ZScsIGhlbHA9J2xvY2FsIGRyeSBydW5zIG9ubHknKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHByaW50KGpzb24uZHVtcHMoYnVpbGQoYXJncy50YXJnZXQsIGFyZ3MuaF9jb21taXQsIGFyZ3Mud29ya3RyZWVfaGFybmVzcyksIGluZGVudD0yKSkK'
SOURCE_BASE64 = '...'
raw = base64.b64decode(SOURCE_BASE64, validate=True)
assert hashlib.sha256(raw).hexdigest() == SOURCE_SHA256, 'Source archive hash mismatch'
namespace = {'__name__': 'bridge_validator', '__file__': '/tmp/bridge-validator/benchmarks/phase2/kaggle/make_bridge_notebook.py'}
exec(compile(base64.b64decode(VALIDATOR_BASE64), '<bridge-validator>', 'exec'), namespace)
manifest, values = namespace['verified_members'](raw, SOURCE_SHA256)
RUN = str(uuid.uuid4())
PRIVATE = Path('/tmp') / ('bridge-' + RUN); PRIVATE.mkdir(mode=0o700)
OUTPUT = Path('/kaggle/working') / ('bridge-' + RUN)
for name, value in values.items():
    destination = PRIVATE.joinpath(*PurePosixPath(name).parts)
    assert destination.resolve().is_relative_to(PRIVATE.resolve())
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open('xb') as handle:
        handle.write(value)
(PRIVATE / 'bundle-manifest.json').write_bytes(json.dumps(manifest, indent=2).encode())
sys.path.insert(0, str(PRIVATE / 'H/benchmarks/phase2/kaggle'))
spec = importlib.util.spec_from_file_location('bridge_study', PRIVATE / 'H/benchmarks/phase2/kaggle/bridge_study.py')
study = importlib.util.module_from_spec(spec); spec.loader.exec_module(study)
print({'run': RUN, 'study': study.STUDY, 'M': manifest['mCommit'][:7], 'H': manifest['hCommit'][:7], 'source_verified': True, 'mode': MODE})

## Chạy
In tiến độ mỗi 30 s. Log, DB và token chỉ nằm trong `/tmp`; ZIP chỉ chứa số đo.

In [ ]:
from IPython.display import display, FileLink
WORK = PRIVATE / 'work'; WORK.mkdir()
summary = study.run_study(PRIVATE / 'M', PRIVATE / 'H', WORK, OUTPUT, RUN, mode=MODE, progress=lambda text: print(text, flush=True))
comparison = summary.get('comparison', {})
print(json.dumps({k: summary.get(k) for k in ('status', 'smoke', 'error_type', 'error')}, indent=2, ensure_ascii=False))
if comparison.get('decision') == 'COMPARED':
    print(f"{'operation':22} {'p95 M (ms)':>16} {'p95 H (ms)':>16} {'H/M p95':>8} {'H/M p50':>8}  đọc")
    for name, op in comparison['operations'].items():
        m, h = (' / '.join(f'{v:.1f}' for v in op['p95_ms'][k]) for k in ('M', 'H'))
        print(f"{name:22} {m:>16} {h:>16} {op['p95_ratio']:8.2f} {op['p50_ratio']:8.2f}  {op['reading']}")
    print(json.dumps({'overall_ratios_H_over_M': comparison['overall_ratios_H_over_M'],
                      'overall_p95': comparison['overall_p95_reading'], 'cpu_per_success': comparison['cpu_per_success_reading']},
                     indent=2, ensure_ascii=False))
else:
    print(json.dumps(comparison, indent=2, ensure_ascii=False))
display(FileLink(summary['download']))

## Gửi kết quả
Tải file `bridge-<run>.zip` ở trên (hoặc tab Output của bản commit) và gửi lại để review.